# Phase 3 | Exploratory data analysis

Explore the approved DrGFreeman Rock-Paper-Scissors release locally. Use the Phase 2 inventory to connect each figure, visual review case and provisional group to its source. This is pre-split curation and description. The dataset attribution and license are recorded in the README.

**Learning goals.** Interpret a distribution with the correct denominator, distinguish an RGB array from its stored encoding, recognize a possible background shortcut, and explain how similarity links can become connected components.

## 1. Identify the inputs and reproduce the run

Select the **Python 3.13 (RPS CNN)** kernel. Run from the repository root or the notebooks directory. The shared run checks the archive, inventory and each eligible file against their approved SHA-256 identities. It uses stable image IDs, seed 42 and the pinned libraries for reproducible selections. Generated evidence and populated review records live in local ignored directories.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "AGENTS.md").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(root))

from src.rps.eda import run_eda

print(sys.executable)
summary, figures, records = run_eda(root)
print(summary["dataset_id"], summary["inventory_sha256"], sep="\n")
print(summary["runtime"])
print(records.relative_to(root))
print(figures.relative_to(root))

def show(name):
    display(Image(filename=str(figures / name)))

## 2. Inventory, class balance and image geometry

The bars count inventory entries. The raw column includes the mirror copy, so eligible counts describe the distinct Phase 2 representatives. Geometry counts include all successfully decoded entries. Width and height describe the file, while NumPy indexes rows before columns. A 300 x 200 RGB image has shape `(200, 300, 3)` and three values per pixel. A single occupied bin confirms a fixed geometry. Class support will inform a later group-aware allocation.

In [ ]:
print("Raw", summary["raw_counts"])
print("Eligible", summary["eligible_counts"])
print("Dispositions", summary["phase2_dispositions"])
print("Format, mode, width, height and channel frequencies", summary["decoded_properties"])
show("inventory.png")

## 3. Color, intensity and lighting

The top plots divide each channel's bin count by the total class pixel count. Brightness uses `0.2126 R + 0.7152 G + 0.0722 B` on stored 8-bit values as a proxy. Per-image P05, P50 and P95 describe dark, central and bright pixels. The lower-left plot shows variation between image means. Across-image box plots summarize those per-image percentiles. Keep these measurements in 0-255 units when checking later scaling assumptions.

The review sheets include the three lowest and three highest mean-brightness images per class. Compare shadows, hand illumination and backgrounds in these examples. The lowest three mean adjacent-pixel gradients per class are quality prompts. Edge strength depends on texture and contrast, so a reviewer evaluates blur and compression artifacts from the images.

In [ ]:
show("distributions.png")
print("Range of per-image mean brightness", summary["brightness_range"])
for label, values in summary["class_statistics"].items():
    print(label, "median image brightness", round(values["median_brightness"], 2))
for sheet in sorted(figures.glob("image-review-*.png")):
    print(sheet.name)

## 4. Background color and possible shortcuts

Use the outer 10 pixels on each side, counting corners once. Green dominance counts pixels where green exceeds both red and blue. Compare per-class border means and green shares. The border approximates the background and may contain parts of hands. In the paired views, gray fills the hidden interior. Border color differences can motivate a later sensitivity check on validation images after the split and baseline exist.

In [ ]:
show("border-color.png")
for label, values in summary["class_statistics"].items():
    print(label, "median border RGB", [round(v, 2) for v in values["median_border_rgb"]],
          "median green share", round(values["median_border_green_share"], 4))
    show(f"border-{label}-01.png")

## 5. Representative gestures, visual label checks and anomalies

Twelve eligible images per class are sampled without replacement after sorting by image ID. Use the contact sheets to review gesture position, orientation, scale, handedness when discernible, framing and clipped fingers. Also check source-label agreement, blur, artifacts and ambiguous gestures. `image-review-*.png` adds the lighting and quality tails and endpoints of competing-label pairs. A row can have several selection reasons. Write observations and decisions against its case ID.

After split freeze, use training examples to assess how resize or padding changes finger shapes and whether translation, zoom, rotation or flips preserve the meaning of the gesture.

In [ ]:
for label in summary["eligible_counts"]:
    show(f"representative-{label}-01.png")
print("Image decisions", summary["review_counts"]["images"])

## 6. Repetition and dHash calibration

Convert each eligible image to grayscale, resize to 9 x 8 with Lanczos, and encode whether each pixel is brighter than its left neighbor. The 64-bit Hamming distance counts positions where two hashes differ. Compare every unordered pair. The initial pilot found 119 pairs at distances 0-2 and 197 more at distance 3. The low tail provides a manageable first queue at radius 2. Include cross-label pairs through distance 4 for priority review. The calibration sheets show two examples per distance through 5, prioritizing cross-label examples when present. Check finger geometry, hand detail, framing and sleeve position. Distance zero means equal dHash values. Exact-content identity comes from the separate Phase 2 file and pixel hashes.

**Pilot visual observations.** D02 has equal hashes, but the wrist cuff and framing differ. D07 at distance 3 shows a fist beside an open hand. D08, also at distance 3, shows similar finger positions and framing in two Paper images. The project owner later confirmed D08 as a near duplicate. D09 and D10 show different gestures at distance 4. These examples illustrate the potential for false matches and missed relationships. The versioned review snapshot records the approved case decisions.

In [ ]:
show("distances.png")
print("All unordered pairs", summary["pair_count"])
print("Pairs at distances 0 through 5", summary["distance_counts"][:6])
print("Review queue by selection", summary["review_counts"]["pair_selection"])
print("Additional boundary review cases", summary["review_counts"]["boundary_pairs"])
for sheet in sorted(figures.glob("distance-examples-*.png")):
    show(sheet.name)

## 7. Manual judgments and group evidence

`image-review.csv`, `pair-review.csv` and `calibration-review.csv` store the project owner's case decisions. The approved waiver in `review-policy.json` retains unreviewed image cases with no unresolved anomaly and adds conservative links for remaining unreviewed same-label candidates. Those CSV decisions remain blank. The versioned snapshot records which images and links the policy covers. Unresolved cross-label candidates and flagged cases still block finalization.

Capture-session evidence includes filename patterns, image metadata keys and archive entry timestamps. A trusted session identifier needs documented meaning and coverage. Exact copies, approved relationships and conservative same-label links form connected components, including paths through excluded rows. When A links to B and B links to C, all three share a group. An unlinked eligible image has a singleton eligible group. The reproducible inventory and group map identify the inputs for a later approved split.

In [ ]:
print("Session evidence", summary["session"])
print("Pair decisions", summary["review_counts"]["pairs"])
print("Grouping and inventory evidence", summary["grouping"])

## 8. Interpret the evidence

- **Measured findings.** Use the inventory totals, observed ranges, class summaries and distance counts above. Reconcile raw, eligible and excluded totals before comparing class distributions.
- **Visual observations.** Record image and pair judgments with their case IDs. Use the border views to note hand intrusion and the lighting tails to describe any shadows. Document examples of false matches near the cutoff.
- **Limitations.** These images share capture conditions. Border statistics provide approximate background evidence, dHash has limited recall, and capture-session uncertainty persists without reliable metadata.
- **Phase 4 questions.** After a grouped split is approved, use training images for resize, RGB scaling and augmentation previews. Consider a planned validation comparison for background sensitivity, based on what the border measurements and visual review show.

Self-directed exercise: compare one pair at distance 0-2 with a pair just outside the cutoff. List the visual features that support or weaken a shared-group judgment.